In [3]:

import pandas as pd

# Load both datasets
noisy = pd.read_csv("potentially_noisy_ratings.csv")
full = pd.read_csv("real_cleaned_dataset.csv")

# Keep the participant and snippet details needed for merging
details = full[
    [
        "participant_code",
        "snippet_id",
        "programming_experience",
        "confidence_1to5",
        "python_comfort_1to5"
    ]
].drop_duplicates(
    subset=["participant_code", "snippet_id"]
)

# Remove existing versions of these columns to avoid duplicates
noisy = noisy.drop(
    columns=[
        "programming_experience",
        "confidence_1to5",
        "python_comfort_1to5"
    ],
    errors="ignore"
)

# Merge using both participant and snippet IDs
result = noisy.merge(
    details,
    on=["participant_code", "snippet_id"],
    how="left",
    validate="many_to_one"
)

# Save the updated dataset
result.to_csv(
    "potentially_noisy_ratings_with_experience_confidence.csv",
    index=False
)

print("Updated dataset saved successfully!")
print("Rows:", len(result))
print("\nColumns:")
print(result.columns.tolist())

print("\nPreview:")
print(result.head())


Updated dataset saved successfully!
Rows: 71

Columns:
['participant_code', 'snippet_id', 'difficulty_1to10', 'snippet_mean', 'deviation', 'z_score', 'programming_experience', 'confidence_1to5', 'python_comfort_1to5']

Preview:
  participant_code snippet_id  difficulty_1to10  snippet_mean  deviation  \
0              705       S007              10.0        3.0125     6.9875   
1              620       S007              10.0        3.0125     6.9875   
2             4873       S007              10.0        3.0125     6.9875   
3              620       S001              10.0        2.5500     7.4500   
4             4873       S001              10.0        2.5500     7.4500   

    z_score programming_experience  confidence_1to5  python_comfort_1to5  
0  3.048722     Less than 6 months              1.0                  1.0  
1  3.048722     Less than 6 months              1.0                  1.0  
2  3.048722     Less than 6 months              1.0                  1.0  
3  2.855350    

In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("outliers_table.csv")

experienced_levels = [
    "1 to 2 years",
    "More than 2 years"
]

high_confidence_levels = [4, 5]

# 3. Identify participants who meet BOTH conditions
experienced = (
    df["programming_experience"]
    .astype(str).str.strip()
    .isin(experienced_levels)
)

confident = pd.to_numeric(
    df["python_comfort_1to5"], errors="coerce"
).isin(high_confidence_levels)

# Participants to exclude
participants_to_remove = df.loc[
    experienced & confident, "participant_code"
].unique()

# 4. Remove ALL rows belonging to those participants
cleaned_df = df[
    ~df["participant_code"].isin(participants_to_remove)
].copy()

# 5. Print the results
print("Original rows:", len(df))
print("Participants excluded:", len(participants_to_remove))
print("Rows removed:", len(df) - len(cleaned_df))
print("Remaining rows:", len(cleaned_df))

print("\nExcluded participant codes:")
print(participants_to_remove)

# 6. Save the filtered dataset
cleaned_df.to_csv(
    "dataset_filtered_by_experience_confidence.csv",
    index=False
)

print("\nFiltered dataset saved successfully!")


Original rows: 71
Participants excluded: 3
Rows removed: 13
Remaining rows: 58

Excluded participant codes:
<StringArray>
['2378_2', '2378', '1445']
Length: 3, dtype: str

Filtered dataset saved successfully!


In [8]:

import pandas as pd

outliers = pd.read_csv("outliers_table.csv")

outliers["difficulty_1to10"] = pd.to_numeric(
    outliers["difficulty_1to10"], errors="coerce"
)
outliers["snippet_mean"] = pd.to_numeric(
    outliers["snippet_mean"], errors="coerce"
)
outliers["python_comfort_1to5"] = pd.to_numeric(
    outliers["python_comfort_1to5"], errors="coerce"
)

print("Total flagged rows:", len(outliers))

print("\nExperience counts:")
print(outliers["programming_experience"].value_counts())

print("\nConfidence counts:")
print(outliers["python_comfort_1to5"].value_counts())

print("\nFlagged ratings of 10:")
print(
    outliers.loc[
        outliers["difficulty_1to10"] == 10,
        [
            "participant_code",
            "snippet_id",
            "difficulty_1to10",
            "snippet_mean",
            "programming_experience",
            "python_comfort_1to5"
        ]
    ].to_string(index=False)
)

print("\nRows meeting experience + confidence criteria:")
print(
    outliers.loc[
        outliers["programming_experience"].isin(
            ["1 to 2 years", "More than 2 years"]
        )
        & outliers["python_comfort_1to5"].isin([4, 5]),
        [
            "participant_code",
            "snippet_id",
            "difficulty_1to10",
            "snippet_mean",
            "programming_experience",
            "python_comfort_1to5"
        ]
    ].to_string(index=False)
)


Total flagged rows: 71

Experience counts:
programming_experience
Less than 6 months    56
More than 2 years     14
1 to 2 years           1
Name: count, dtype: int64

Confidence counts:
python_comfort_1to5
1.0    45
2.0    11
5.0     8
4.0     5
3.0     2
Name: count, dtype: int64

Flagged ratings of 10:
participant_code snippet_id  difficulty_1to10  snippet_mean programming_experience  python_comfort_1to5
             705       S007              10.0        3.0125     Less than 6 months                  1.0
             620       S007              10.0        3.0125     Less than 6 months                  1.0
            4873       S007              10.0        3.0125     Less than 6 months                  1.0
             620       S001              10.0        2.5500     Less than 6 months                  1.0
            4873       S001              10.0        2.5500     Less than 6 months                  1.0
             705       S001              10.0        2.5500     Less 

In [9]:
import pandas as pd

# Load the original dataset
actual = pd.read_csv("real_cleaned_dataset.csv")

# Load the flagged ratings
outliers = pd.read_csv("outliers_table.csv")

# Select the experienced, confident participants' flagged ratings
selected = outliers[
    (outliers["programming_experience"].isin([
        "1 to 2 years",
        "More than 2 years"
    ]))
    & (outliers["python_comfort_1to5"].isin([4, 5]))
].copy()

# Remove only the selected participant-snippet ratings
remove_pairs = selected[["participant_code", "snippet_id"]].drop_duplicates()

cleaned = actual.merge(
    remove_pairs.assign(remove=True),
    on=["participant_code", "snippet_id"],
    how="left"
)

cleaned = cleaned[cleaned["remove"].isna()].drop(columns="remove")

print("Original rows:", len(actual))
print("Selected flagged ratings:", len(remove_pairs))
print("Rows removed:", len(actual) - len(cleaned))
print("Remaining rows:", len(cleaned))

cleaned.to_csv("actual_dataset_without_selected_outliers.csv", index=False)

Original rows: 1040
Selected flagged ratings: 13
Rows removed: 13
Remaining rows: 1027


In [11]:
import pandas as pd

# 1. Load the original dataset (1,040 rows)
actual = pd.read_csv("real_cleaned_dataset.csv")

# 2. Load the CSV containing the 71 flagged outliers
outliers = pd.read_csv("strong_potentially_noisy_ratings.csv")

# 3. Select the 71 outliers you previously identified
# If this file contains exactly 71 rows, use all of them.
outlier_pairs = outliers[["participant_code", "snippet_id"]].drop_duplicates()

# 4. Remove those participant-snippet ratings from the original dataset
original_rows = len(actual)

cleaned = actual.merge(
    outlier_pairs.assign(is_outlier=True),
    on=["participant_code", "snippet_id"],
    how="left"
)

cleaned = cleaned[cleaned["is_outlier"].isna()].drop(
    columns=["is_outlier"]
)

# 5. Display the results
print("Original dataset rows:", original_rows)
print("Flagged outlier rows in file:", len(outliers))
print("Unique participant-snippet pairs:", len(outlier_pairs))
print("Rows removed:", original_rows - len(cleaned))
print("Remaining rows:", len(cleaned))

# 6. Save the cleaned dataset
cleaned.to_csv("actual_dataset_without_71_outliers.csv", index=False)

print("\nSaved: actual_dataset_without_71_outliers.csv")

Original dataset rows: 1040
Flagged outlier rows in file: 21
Unique participant-snippet pairs: 21
Rows removed: 21
Remaining rows: 1019

Saved: actual_dataset_without_71_outliers.csv
